# Experiment 17 — Resolution kinetics, a pro-resolving adjunct and community memory

Addresses the colleague's review of 23 Oct 2026 (points 1 and 3) and the size of the rechallenge effect.

- **17A Resolution rate as a host trait.** Inflammation rises as before but resolves at rate ρ: dH/dt = k (F − H), with k = 1 while F ≥ H and k = ρ while F < H. Equilibria, folds and the instant-host control are unchanged, so every difference is kinetic. ρ = 0.25–4 (ρ = 1 is the archived model). Outcomes: memory acquired in one experimental-gingivitis episode and the vulnerability it leaves (A1); therapy, post-therapy window, half-life and relapse ceiling, at r = 1 and 0.03 (A2); calibrated 21-day rechallenge (A3); ratchet threshold under periodic plaque removal (A4).
- **17B Pro-resolving adjunct.** A course of L = 28 days that adds a clearance term −λH (λ = 0.25–1; λ = 1 halves H at a given signal while it is given), standing for a lipoxin-analogue rinse. B1: relapse ceiling when the course starts 0, 30, 60 or 90 days after therapy, at r = 1 and 0.03 (the factorial of Table 1, row 8). B2: a 21-day challenge at 0–365 days after therapy, with no course, a course concurrent with the challenge, or an early course (days 0–28) before it.
- **17C Community memory.** A slow community state B lowers the half-saturation of community support, G(H; K e^{−B}), relaxes towards βM at rate μ, and is **not** reset by debridement. An *instant community* (B = βM at every step) shares every equilibrium, so the design is 2 × 2: host memory (ε = 0.01, 0.03, or instant host) × community memory (μ = 0.1, 0.03, 0.01, or instant community), at β = 0.25 and 0.5 and r = 1 and 0.03. C1: therapy, retention, relapse ceiling. C2: critical challenge and calibrated rechallenge at 0–365 days after therapy against naive sites and against naive sites given the treated site's community state (the composition-matched split-mouth comparison); decomposition of the treated-site deficit into host, community and interaction.

Every extension reduces exactly to `periomemory` 1.0.0 when ρ = 1, λ = 0 and β = 0; section 0 verifies this on trajectories and reproduces the Experiment_14 anchors through the extended model. All extensions keep the system cooperative (ρ and λ only scale or add a self-term; B and M reinforce each other), so conversion stays monotone in challenge size and bisection is valid.

Set `FAST = True` for a smoke test; `FAST = False` for the full run. On Colab the export folder is written to Google Drive (`MyDrive/Biosemiotic_Perio`) as the run proceeds, with a checkpoint after every part. **If the runtime disconnects, reconnect and choose Run all again**: finished parts are skipped. The folder is zipped at the end.

In [ ]:
# Install the archived model package (skip if already installed)
import importlib, subprocess, sys
try:
    import periomemory  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/CarloGalliUnipr/periodontal-host-memory-model.git"], check=True)

In [ ]:
import json, os, platform, shutil, time, itertools, glob
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import brentq
import periomemory as pm
from periomemory.model import hill

# ----------------------------------------------------------------- run settings
FAST = False                # True: smoke test with reduced grids
TARGET_NAIVE_PEAK = 0.30    # naive peak H that stands for experimental gingivitis (as in 15C)
SEED = 20261023

# ----------------------------------------------------------------- where results are saved
USE_DRIVE = True            # False: write to the Colab disk only (lost on disconnection)
DRIVE_DIR = "/content/drive/MyDrive/Biosemiotic_Perio"
RESUME = True               # False: delete any previous export and start from scratch
NAME = "Experiment_17_Resolution_Community_Memory_EXPORT" + ("_FAST" if FAST else "")
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    OUT = os.path.join(DRIVE_DIR, NAME)
else:
    OUT = NAME
if os.path.exists(OUT) and not RESUME:
    shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)
print("export folder:", os.path.abspath(OUT))
CHECKS = []
T0 = time.time()
_START = [0]

def part_dir(name):
    d = os.path.join(OUT, name); os.makedirs(d, exist_ok=True); return d

def done(part, tag):
    return os.path.exists(os.path.join(OUT, part, f"_DONE_{tag}"))

def start_part():
    _START[0] = len(CHECKS)

def mark_done(part, tag):
    new = CHECKS[_START[0]:]
    pd.DataFrame(new, columns=["module", "check", "value", "passed"]).to_csv(
        os.path.join(part_dir(part), f"checks_{tag}.csv"), index=False)
    for c in new:
        c["saved"] = True
    with open(os.path.join(part_dir(part), f"_DONE_{tag}"), "w") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S"))
    print(f"[checkpoint] {part} {tag} saved  ({time.time() - T0:.0f} s)")

def load(part, fname):
    return pd.read_csv(os.path.join(OUT, part, fname))

def save(df, part, fname):
    df.to_csv(os.path.join(part_dir(part), fname), index=False)
    return df

def check(part, what, value, passed=None):
    CHECKS.append(dict(module=part, check=what, value=value, passed=passed, saved=False))
    print(f"[{part}] {what}: {value}" + ("" if passed is None else f"  -> {'OK' if passed else 'FAILED'}"))

# ----------------------------------------------------------------- parameters
REF = dict(pm.REFERENCE)                      # n=q=4, K=0.9, th0=0.8, r=1, a=1
UB_REF = 0.20
EPS_LIST = [0.03, 0.01]
R_LIST = [1.0, 0.03] if not FAST else [1.0]
DAYS = np.array([0, 15, 30, 60, 90, 180, 365], float) if not FAST else np.array([0, 30, 90, 365], float)
NBIS = 16 if not FAST else 11
RHOS = [0.25, 0.5, 1.0, 2.0, 4.0] if not FAST else [0.5, 1.0, 2.0]
REC_LIST = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0] if not FAST else [0.0, 0.5, 1.0, 2.0]
LAMS = [0.25, 0.5, 1.0] if not FAST else [0.5, 1.0]
COURSE_DAYS = 28.0
STARTS = [0.0, 30.0, 60.0, 90.0] if not FAST else [0.0, 60.0]
BETAS = [0.25, 0.5] if not FAST else [0.5]
MUS = [0.1, 0.03, 0.01] if not FAST else [0.03]
UB_STEP = 0.005 if not FAST else 0.02
PEAK_STEP = 0.005 if not FAST else 0.02
SETTLE = 1000.0                               # 10 / 0.01: slowest memory used
DUR21 = pm.align(pm.days_to_units(21.0))

PARAMS = dict(FAST=FAST, TARGET_NAIVE_PEAK=TARGET_NAIVE_PEAK, SEED=SEED)
pfile = os.path.join(OUT, "run_settings.json")
if os.path.exists(pfile):
    prev = json.load(open(pfile))
    if prev != PARAMS:
        raise RuntimeError(f"The export folder holds a run with different settings {prev}; "
                           "set RESUME = False to start again, or restore the settings.")
json.dump(PARAMS, open(pfile, "w"))
with open(os.path.join(OUT, "parameters.json"), "w") as f:
    json.dump(dict(PARAMS, REF=REF, UB_REF=UB_REF, EPS_LIST=EPS_LIST, R_LIST=R_LIST, DAYS=DAYS.tolist(),
                   RHOS=RHOS, REC_LIST=REC_LIST, LAMS=LAMS, COURSE_DAYS=COURSE_DAYS, STARTS=STARTS,
                   BETAS=BETAS, MUS=MUS, SETTLE=SETTLE, K_ACCESS=pm.K_ACCESS, T_PC=pm.T_PC,
                   UNIT_DAYS=pm.UNIT_DAYS), f, indent=1)
with open(os.path.join(OUT, "software_versions.txt"), "w") as f:
    f.write(f"python {platform.python_version()}\nnumpy {np.__version__}\nscipy {scipy.__version__}\n"
            f"pandas {pd.__version__}\nmatplotlib {matplotlib.__version__}\nperiomemory {pm.__version__}\n")

## Extended model

`ExtModel` integrates (H, M, A, D, B) with the same fixed-step RK4 as `periomemory.Model`:

- dH/dt = k(F, H) (F − H) − λ(t) H, with k = 1 while F ≥ H and k = ρ while F < H
- dM/dt = r (−M + G(H; K e^{−B}) + γ D (c + (1 − c) H))
- dA/dt = ε (A\*(H) − A); instant host: A = A\*(H)
- dD/dt = δ max(H − h_D, 0)(1 − D)
- dB/dt = μ (βM − B); instant community: B = βM

Every parameter, including the two `instant` flags, can be a vector over sites, so whole designs run in one call. Debridement sets M to 0 and leaves B (it sets B to 0 only in the instant community, where B = βM). With community memory, equilibria satisfy M = G(H; K e^{−βM}); `u_curve_B`, `folds_B` and `eq_state_B` solve this for the smallest root and report a gap if a second root exists.

In [ ]:
class ExtModel:
    def __init__(self, n, q, K, th0, r, a, b=0.0, eps=0.03, instant=False, gamma=0.0, c=1.0,
                 delta=0.0, hD=pm.H_D, rho=1.0, beta=0.0, mu=0.03, instant_community=False,
                 lam_fun=None, dt=pm.DT):
        self.n, self.q, self.K, self.th0, self.r, self.a, self.b = [
            np.asarray(v, float) for v in (n, q, K, th0, r, a, b)]
        self.eps, self.gamma, self.c, self.delta, self.hD = [
            np.asarray(v, float) for v in (eps, gamma, c, delta, hD)]
        self.rho, self.beta, self.mu = [np.asarray(v, float) for v in (rho, beta, mu)]
        self.inst = np.asarray(instant, bool); self.icom = np.asarray(instant_community, bool)
        self.any_inst, self.any_icom = bool(self.inst.any()), bool(self.icom.any())
        self.lam_fun = lam_fun
        self.dt = float(dt)

    def target(self, H):
        return self.a * H - self.b * H * H

    def deriv(self, H, M, A, D, B, u, lam):
        if self.any_inst:
            A = np.where(self.inst, self.target(H), A)
        if self.any_icom:
            B = np.where(self.icom, self.beta * M, B)
        F = hill(M + u, self.th0 * np.exp(-A), self.n)
        k = np.where(F < H, self.rho, 1.0)
        dH = k * (F - H) - lam * H
        dM = self.r * (-M + hill(H, self.K * np.exp(-B), self.q)
                       + self.gamma * D * (self.c + (1.0 - self.c) * H))
        dA = self.eps * (self.target(H) - A)
        if self.any_inst:
            dA = np.where(self.inst, 0.0, dA)
        dD = self.delta * np.maximum(H - self.hD, 0.0) * (1.0 - D)
        dB = self.mu * (self.beta * M - B)
        if self.any_icom:
            dB = np.where(self.icom, 0.0, dB)
        return dH, dM, dA, dD, dB

    def _lam(self, t):
        return 0.0 if self.lam_fun is None else self.lam_fun(t)

    def run(self, state, u_fun, t0, t1, monitor=None):
        dt = self.dt
        X = [np.array(x, float) for x in state]
        if len(X) == 4:
            X.append(np.zeros_like(X[0]))
        n = int(round((t1 - t0) / dt)); t = t0
        for _ in range(n):
            u1, u2, u3 = u_fun(t), u_fun(t + dt / 2), u_fun(t + dt)
            l1, l2, l3 = self._lam(t), self._lam(t + dt / 2), self._lam(t + dt)
            k1 = self.deriv(*X, u1, l1)
            k2 = self.deriv(*[x + dt / 2 * k for x, k in zip(X, k1)], u2, l2)
            k3 = self.deriv(*[x + dt / 2 * k for x, k in zip(X, k2)], u2, l2)
            k4 = self.deriv(*[x + dt * k for x, k in zip(X, k3)], u3, l3)
            X = [x + dt / 6 * (a1 + 2 * a2 + 2 * a3 + a4) for x, a1, a2, a3, a4 in zip(X, k1, k2, k3, k4)]
            t = t + dt
            if self.any_inst:
                X[2] = np.where(self.inst, self.target(X[0]), X[2])
            if self.any_icom:
                X[4] = np.where(self.icom, self.beta * X[1], X[4])
            if monitor is not None:
                monitor(t, *X)
        return tuple(X), t


def ext(p, eps=0.03, instant=False, **kw):
    return ExtModel(**p, eps=eps, instant=instant, **kw)


def course(start_units, lam0, L_units):
    # Per-site clearance course: lam0 between start and start + L (time 0 = end of plaque control)
    s = np.asarray(start_units, float); l0 = np.asarray(lam0, float)
    return lambda t: np.where((t >= s - 1e-9) & (t < s + L_units - 1e-9), l0, 0.0)


# ----------------------------------------------------------------- equilibria with community memory
HG = pm.equilibria.HG
_UC = {}

def community_M(p, H, beta, tol=1e-13, maxit=5000):
    # Smallest and largest M solving M = G(H; K e^{-beta M})
    H = np.asarray(H, float)
    lo = hill(H, p["K"], p["q"]); hi = np.ones_like(H)
    for _ in range(maxit):
        lo_n = hill(H, p["K"] * np.exp(-beta * lo), p["q"])
        hi_n = hill(H, p["K"] * np.exp(-beta * hi), p["q"])
        if np.max(np.abs(lo_n - lo)) < tol and np.max(np.abs(hi_n - hi)) < tol:
            lo, hi = lo_n, hi_n; break
        lo, hi = lo_n, hi_n
    return lo, hi

def u_curve_B(p, beta, H=None, a=None):
    a = p["a"] if a is None else a
    if H is None:
        key = (tuple(sorted(p.items())), float(beta), float(a))
        if key not in _UC:
            M, Mhi = community_M(p, HG, beta)
            _UC[key] = (p["th0"] * np.exp(-a * HG) * (HG / (1 - HG)) ** (1 / p["n"]) - M, float(np.max(Mhi - M)))
        return _UC[key]
    M, Mhi = community_M(p, H, beta)
    return p["th0"] * np.exp(-a * H) * (H / (1 - H)) ** (1 / p["n"]) - M, float(np.max(Mhi - M))

def folds_B(p, beta, a=None):
    u, gap = u_curve_B(p, beta, a=a)
    du = np.diff(u)
    idx = np.where(np.sign(du[1:]) != np.sign(du[:-1]))[0] + 1
    if len(idx) < 2:
        return None, gap
    return (HG[idx[0]], u[idx[0]], HG[idx[1]], u[idx[1]]), gap

def eq_state_B(p, ub, branch, beta):
    # Equilibrium (H, M, A, D, B) at constant plaque ub, D = 0; None if the branch is absent
    u, _ = u_curve_B(p, beta)
    v = u - ub
    i = np.where(np.sign(v[1:]) != np.sign(v[:-1]))[0]
    if not len(i):
        return None
    j = i[0] if branch == "low" else i[-1]
    f = lambda H: u_curve_B(p, beta, np.array([H]))[0][0] - ub
    H = brentq(f, HG[j], HG[j + 1], xtol=1e-15)
    M = float(community_M(p, np.array([H]), beta)[0][0])
    return float(H), M, float(p["a"] * H), 0.0, float(beta * M)

def tile5(state, N):
    return tuple(np.full(N, float(x)) for x in state)

def cat5(states):
    return tuple(np.concatenate([np.atleast_1d(s[k]) for s in states]) for k in range(5))

## Shared protocol functions

- `therapy5`: debridement (M → 0, B kept) and `t_pc` units of perfect plaque control; returns the state, H at the end of plaque control, and the fractions of A and B retained.
- `eg5`: the 21-day experimental-gingivitis challenge of 14A/15C (plaque rising from baseline towards U at 0.5 per unit), peak H in the window, persistence (H > 0.5) after an optional settling period at baseline. Time 0 is the start of the call (end of plaque control for treated sites), so a clearance course defined by `course` is timed from the same origin.
- Critical challenges use `pm.critical_challenge` (step challenge of duration 2/ε, settle 10/ε, as in Figure 3D; duration 20 and settle 200 for the instant host alone). A 21-day challenge is used only for peak responses, because naive memory hosts recover from it at any plateau (15C).
- `calibrate_vec`: plateau U\* giving a naive peak of TARGET_NAIVE_PEAK, per site.
- `relapse_ceiling_from`: lowest baseline from which every higher baseline relapses without challenge.
- `ratchet_ext`: periodic regrowth and removal from a naive start (as `pm.ratchet`), returning the conversion cycle.

In [ ]:
def therapy5(m, states, t_pc=pm.T_PC):
    H, M, A, D, B = [np.array(x, float) for x in (list(states) + [np.zeros_like(states[0])] * (5 - len(states)))]
    A0, B0 = A.copy(), B.copy()
    M = np.zeros_like(M)
    st, _ = m.run((H, M, A, D, B), pm.const(0.0), 0.0, t_pc)
    with np.errstate(invalid="ignore", divide="ignore"):
        retA = np.where(A0 != 0, st[2] / A0, np.nan)
        retB = np.where(B0 != 0, st[4] / B0, np.nan)
    return st, st[0].copy(), retA, retB

def _bc(x, N):
    return np.broadcast_to(np.asarray(x, float), (N,)).copy()

def eg5(m, states, ub, t_on, U, dur=DUR21, ka=pm.K_ACCESS, settle=0.0):
    X = [np.array(x, float) for x in states]
    if len(X) == 4:
        X.append(np.zeros_like(X[0]))
    N = X[0].size
    ub, U = _bc(ub, N), _bc(U, N); t_on = pm.align(_bc(t_on, N))
    sched = pm.schedule(ub, t_on, dur, U, ka=ka)
    peak = np.zeros(N)
    def mon(t, H, M, A, D, B):
        win = (t > t_on + 1e-9) & (t <= t_on + dur + 1e-9)
        np.maximum(peak, np.where(win, H, 0.0), out=peak)
    t_end = float((t_on + dur).max() + pm.days_to_units(14))
    st, _ = m.run(X, sched, 0.0, t_end, monitor=mon)
    if settle > 0:
        st, _ = m.run(st, pm.const(ub), t_end, t_end + settle)
    return dict(peak=peak, persistent=st[0] > 0.5, state=st)

def calibrate_vec(m, nv, ub, target=TARGET_NAIVE_PEAK, hi=2.0, it=30):
    N = np.asarray(nv[0]).size
    lo = _bc(ub, N); hi_ = np.full(N, hi)
    pk = lambda U: eg5(m, nv, ub, 0.0, U)["peak"]
    ok = pk(hi_) >= target
    for _ in range(it):
        mid = 0.5 * (lo + hi_); below = pk(mid) < target
        lo = np.where(below, mid, lo); hi_ = np.where(below, hi_, mid)
    return np.where(ok, hi_, np.nan)

def relapse_ceiling_from(ubs, relapsed):
    relapsed = np.asarray(relapsed, bool)
    for i in range(len(ubs)):
        if relapsed[i:].all():
            return float(ubs[i])
    return np.nan

def ratchet_ext(m, peaks, ncyc, T=60.0, k=pm.K_REGROW):
    peaks = np.asarray(peaks, float); N = len(peaks)
    umax = peaks / (1 - np.exp(-k * T))
    st = tuple(np.zeros(N) for _ in range(5))
    u_fun = lambda t: umax * (1 - np.exp(-k * t))
    ent = np.zeros((ncyc, N), bool)
    for c in range(ncyc):
        hmin = np.full(N, 9.0)
        def mon(t, H, M, A, D, B): np.minimum(hmin, H, out=hmin)
        st, _ = m.run(st, u_fun, 0.0, T, monitor=mon)
        ent[c] = hmin > 0.5
    tail = np.flip(np.logical_and.accumulate(np.flip(ent, 0), 0), 0)
    return np.where(tail[-1], np.argmax(tail, 0) + 1.0, np.nan)

def threshold_from(peaks, conv):
    ok = ~np.isnan(conv)
    for i in range(len(ok)):
        if ok[i:].all():
            return float(peaks[i])
    return np.nan

HOSTS = [(0.03, False, "eps=0.03"), (0.01, False, "eps=0.01"), (0.03, True, "instant")]

## 0. The extension reduces to `periomemory` 1.0.0

Trajectories of `ExtModel` with ρ = 1, λ = 0, β = 0 are compared with `pm.Model` (memory and instant host, with attachment loss and a challenge), and the Experiment_14 anchors are recomputed through `ExtModel`.

In [ ]:
P = "0_reduction"
start_part()
st0 = pm.tile(pm.eq_state(REF, 0.2, "high", D=0.1), 3)
u_ = pm.schedule(0.2, np.array([5.0, 10.0, 20.0]), 15.0, np.array([0.3, 0.5, 0.7]), ka=0.5)
for inst in (False, True):
    a_, _ = pm.Model(**REF, eps=0.01, instant=inst, gamma=pm.GAMMA, delta=0.001).run(st0, u_, 0, 150)
    b_, _ = ext(REF, 0.01, inst, gamma=pm.GAMMA, delta=0.001).run(st0, u_, 0, 150)
    dmax = max(float(np.max(np.abs(x - y))) for x, y in zip(a_, b_[:4]))
    check(P, f"max |ExtModel - pm.Model| over 1500 steps (instant={inst})", dmax, dmax == 0.0)
for ub_, br in [(0.2, "low"), (0.2, "high")]:
    e0, e1 = pm.eq_state(REF, ub_, br, gamma=0.0), eq_state_B(REF, ub_, br, 0.0)
    check(P, f"eq_state_B(beta=0) equals pm.eq_state ({br})", abs(e0[0] - e1[0]), abs(e0[0] - e1[0]) < 1e-8)

m = ext(REF, 0.01)
tr, Hend, retA, _ = therapy5(m, tile5(eq_state_B(REF, UB_REF, "high", 0.0), 1))
check(P, "A retained after therapy, eps=0.01 (14A: 0.970567)", round(float(retA[0]), 6), abs(retA[0] - 0.970567) < 1e-5)
r_ = eg5(m, tr, UB_REF, pm.days_to_units(90.0), 0.40)
check(P, "treated peak H at 90 days, U=0.40 (14A: 0.527179)", round(float(r_["peak"][0]), 6), abs(r_["peak"][0] - 0.527179) < 1e-5)
r_ = eg5(m, tile5(eq_state_B(REF, UB_REF, "low", 0.0), 1), UB_REF, pm.days_to_units(90.0), 0.40)
check(P, "naive peak H, U=0.40 (14A: 0.060207)", round(float(r_["peak"][0]), 6), abs(r_["peak"][0] - 0.060207) < 1e-5)
m = ext(REF, 0.03)
both = cat5([tile5(eq_state_B(REF, UB_REF, "low", 0.0), 1), therapy5(m, tile5(eq_state_B(REF, UB_REF, "high", 0.0), 1))[0]])
c_, _ = pm.critical_challenge(m, both, UB_REF, np.zeros(2), np.full(2, 2 / 0.03), 10 / 0.03, n_iter=16)
check(P, "critical challenge eps=0.03, 2/eps, naive / treated (14B: 0.564890 / 0.294977)",
      (round(float(c_[0]), 6), round(float(c_[1]), 6)), abs(c_[0] - 0.564890) < 2e-6 and abs(c_[1] - 0.294977) < 2e-6)
mark_done(P, "0")

## 17A. Resolution rate as a host trait

**A1. One gingivitis episode.** Naive sites receive a 21-day challenge at the plateau U\* that gives a naive peak of 0.30 at ρ = 1 (same U\* for every ρ, so only resolution differs). Recorded: peak H, memory A at the end of the window and its maximum, ∫H dt, and the time H stays above 0.1 after the challenge ends. The critical step challenge (duration 2/ε, settle 10/ε, as in Figure 3D) 0, 30 and 90 days after the end of the first episode (window + 14 days) is compared with that of a naive site with the same ρ. (A 21-day challenge cannot be used for this: naive memory hosts recover from it at any plateau, as 15C showed.)

**A2. After therapy.** Defined therapy at ρ; H at the end of plaque control, A retained, restoration; critical step challenge of duration 2/ε (as Figure 3D–E) at recovery εT ∈ REC_LIST for treated and naive sites, and vulnerability half-life; instant host (duration 20) for reference; relapse ceiling. r = 1 and 0.03.

**A3. Calibrated rechallenge** (r = 1): U\* of the ρ = 1 naive site, treated and naive sites at DAYS after therapy, peak H and persistence.

**A4. Ratchet threshold** at T = 60 for ε = 0.01 (67 intervals) and the instant host (20 intervals), by ρ. ρ = 1 must reproduce `pm.ratchet_threshold` on the same grid.

In [ ]:
P = "17A_resolution_rate"
RHO = np.array(RHOS, float); nR = len(RHO)
if not done(P, "A1"):
    start_part()
    rows = []
    lowst = eq_state_B(REF, UB_REF, "low", 0.0)
    delays = np.array([0.0, 30.0, 90.0])
    for eps in EPS_LIST:
        Ustar = float(calibrate_vec(ext(REF, eps), tile5(lowst, 1), UB_REF)[0])
        m = ext(REF, eps, rho=RHO)
        nv = tile5(lowst, nR)
        integ = np.zeros(nR); amax = np.zeros(nR); last_inf = np.zeros(nR); peak = np.zeros(nR)
        t_win_end = float(DUR21)
        def mon(t, H, M, A, D, B):
            integ[:] += H * pm.DT; np.maximum(amax, A, out=amax)
            if t <= t_win_end + 1e-9:
                np.maximum(peak, H, out=peak)
            else:
                last_inf[:] = np.where(H > 0.1, t - t_win_end, last_inf)
        t_end = float(DUR21 + pm.days_to_units(14))
        st, _ = m.run(nv, pm.schedule(UB_REF, 0.0, DUR21, Ustar, ka=pm.K_ACCESS), 0.0, t_end, monitor=mon)
        A_end = st[2].copy()
        # second challenge after delays: treated-by-episode vs naive, same rho
        nd = len(delays)
        m2 = ext(REF, eps, rho=np.tile(np.repeat(RHO, nd), 2))
        sts = cat5([tuple(np.repeat(x, nd) for x in st), tuple(np.repeat(x, nd) for x in nv)])
        tons = np.tile(pm.days_to_units(delays), 2 * nR)
        c_, clo = pm.critical_challenge(m2, sts, UB_REF, tons, np.full(tons.size, 2 / eps), 10 / eps, n_iter=NBIS)
        c_ep, c_nv = c_[:nR * nd].reshape(nR, nd), c_[nR * nd:].reshape(nR, nd)
        for i, rho in enumerate(RHO):
            row = dict(eps=eps, rho=rho, U=Ustar, peak_H=peak[i], A_end_episode=A_end[i], A_max=amax[i],
                       H_integral=integ[i], days_H_above_0p1_after_challenge=last_inf[i] * pm.UNIT_DAYS)
            for j, d in enumerate(delays):
                row[f"crit_after_episode_{int(d)}d"] = c_ep[i, j]; row[f"crit_naive_{int(d)}d"] = c_nv[i, j]
                row[f"ratio_{int(d)}d"] = c_ep[i, j] / c_nv[i, j]
            rows.append(row)
    A1 = save(pd.DataFrame(rows), P, "A1_gingivitis_episode.csv")
    print(A1.round(4).to_string())
    for eps in EPS_LIST:
        g = A1[A1.eps == eps].sort_values("rho")
        check(P, f"A at end of episode falls as rho rises (eps={eps})", g.A_end_episode.round(4).tolist(),
              bool(np.all(np.diff(g.A_end_episode.values) <= 1e-9)))
    mark_done(P, "A1")

In [ ]:
if not done(P, "A2"):
    start_part()
    rows, wrows, crow = [], [], []
    REC = np.array(REC_LIST, float); nE = len(REC)
    for r in R_LIST:
        p = dict(REF, r=r)
        hist = tile5(eq_state_B(p, UB_REF, "high", 0.0), nR)
        lowst = eq_state_B(p, UB_REF, "low", 0.0)
        for eps, inst, host in HOSTS:
            m = ext(p, eps, inst, rho=RHO)
            st, Hend, retA, _ = therapy5(m, hist)
            fin, _ = m.run(st, pm.const(UB_REF), 0.0, SETTLE if not inst else 200.0)
            for i, rho in enumerate(RHO):
                rows.append(dict(r=r, host=host, rho=rho, H_after_plaque_control=Hend[i], A_retained=retA[i],
                                 restored=bool(fin[0][i] < 0.5)))
            if inst:
                both = cat5([tile5(lowst, nR), st])
                mi = ext(p, eps, True, rho=np.tile(RHO, 2))
                c_, clo = pm.critical_challenge(mi, both, UB_REF, np.zeros(2 * nR), np.full(2 * nR, 20.0), 200.0, n_iter=NBIS)
                for i, rho in enumerate(RHO):
                    crow.append(dict(r=r, rho=rho, crit_instant_naive=c_[i], crit_instant_treated=c_[nR + i]))
                continue
            # window: step challenges of 2/eps after recovery eps*T
            sts = cat5([tuple(np.repeat(x, nE) for x in tile5(lowst, nR)), tuple(np.repeat(x, nE) for x in st)])
            mm = ext(p, eps, rho=np.tile(np.repeat(RHO, nE), 2))
            tons = np.tile(REC / eps, 2 * nR)
            c_, clo = pm.critical_challenge(mm, sts, UB_REF, tons, np.full(2 * nR * nE, 2 / eps), 10 / eps, n_iter=NBIS)
            cn, ct = c_[:nR * nE].reshape(nR, nE), c_[nR * nE:].reshape(nR, nE)
            for i, rho in enumerate(RHO):
                V = (cn[i] - ct[i]) / (cn[i, 0] - ct[i, 0])
                for j, e in enumerate(REC):
                    wrows.append(dict(r=r, host=host, eps=eps, rho=rho, rec_memory_units=e, crit_naive=cn[i, j],
                                      crit_treated=ct[i, j], ratio=ct[i, j] / cn[i, j], V=V[j]))
    A2 = save(pd.DataFrame(rows), P, "A2_therapy.csv")
    A2w = save(pd.DataFrame(wrows), P, "A2_window_vs_recovery.csv")
    A2i = save(pd.DataFrame(crow), P, "A2_instant_host.csv")
    hl = []
    for (r, host, rho), g in A2w.groupby(["r", "host", "rho"]):
        g = g.sort_values("rec_memory_units")
        hl.append(dict(r=r, host=host, rho=rho, ratio_immediately=g.ratio.iloc[0],
                       half_life_memory_times=pm.half_life(g.rec_memory_units, g.V)))
    A2h = save(pd.DataFrame(hl), P, "A2_half_life.csv")
    print(A2.round(4).to_string()); print(A2h.round(3).to_string())
    g = A2w[(A2w.r == 1.0) & (A2w.host == "eps=0.03") & (A2w.rho == 1.0) & (A2w.rec_memory_units == 0)]
    if len(g):
        check(P, "rho=1 reproduces 14B treated/naive at eps=0.03 (0.294977/0.564890 = 0.5222)",
              round(float(g.ratio.iloc[0]), 4),
              abs(float(g.ratio.iloc[0]) - 0.294977 / 0.564890) < max(1e-4, 4 * 3.8 / 2 ** NBIS / 0.56))
    mark_done(P, "A2")

In [ ]:
if not done(P, "A2_relapse"):
    start_part()
    fold = pm.folds(REF)[1]
    ubs = np.round(np.arange(0.15, fold + 1e-9, UB_STEP), 4); nU = len(ubs)
    rows = []
    for r in R_LIST:
        p = dict(REF, r=r)
        his = [eq_state_B(p, u, "high", 0.0) for u in ubs]
        hist = tuple(np.tile(np.array([h[k] for h in his]), nR) for k in range(5))
        ubv = np.tile(ubs, nR)
        for eps, inst, host in HOSTS:
            m = ext(p, eps, inst, rho=np.repeat(RHO, nU))
            st, _, _, _ = therapy5(m, hist)
            fin, _ = m.run(st, pm.const(ubv), 0.0, SETTLE if not inst else 200.0)
            rel = (fin[0] > 0.5).reshape(nR, nU)
            for i, rho in enumerate(RHO):
                c = relapse_ceiling_from(ubs, rel[i])
                rows.append(dict(r=r, host=host, rho=rho, relapse_ceiling=c, static_fold=fold, ceiling_over_fold=c / fold))
    A2r = save(pd.DataFrame(rows), P, "A2_relapse_ceiling.csv")
    print(A2r.round(4).to_string())
    mark_done(P, "A2_relapse")

In [ ]:
if not done(P, "A3"):
    start_part()
    rows = []
    lowst = eq_state_B(REF, UB_REF, "low", 0.0); hist = eq_state_B(REF, UB_REF, "high", 0.0)
    nD = len(DAYS)
    for eps, inst, host in HOSTS:
        Ustar = float(calibrate_vec(ext(REF, eps, inst), tile5(lowst, 1), UB_REF)[0])
        m = ext(REF, eps, inst, rho=RHO)
        tr, _, _, _ = therapy5(m, tile5(hist, nR))
        sts = cat5([tuple(np.repeat(x, nD) for x in tile5(lowst, nR)), tuple(np.repeat(x, nD) for x in tr)])
        mm = ext(REF, eps, inst, rho=np.tile(np.repeat(RHO, nD), 2))
        res = eg5(mm, sts, UB_REF, np.tile(pm.days_to_units(DAYS), 2 * nR), Ustar, settle=SETTLE if not inst else 200.0)
        for h, hname in enumerate(["naive", "treated"]):
            for i, rho in enumerate(RHO):
                for j, d in enumerate(DAYS):
                    k = h * nR * nD + i * nD + j
                    rows.append(dict(host=host, rho=rho, U_star_rho1=Ustar, history=hname, days=d,
                                     peak_H=res["peak"][k], persistent=bool(res["persistent"][k])))
    A3 = save(pd.DataFrame(rows), P, "A3_rechallenge.csv")
    print(A3.pivot_table(index=["host", "history", "rho"], columns="days", values="peak_H").round(2).to_string())
    mark_done(P, "A3")

In [ ]:
if not done(P, "A4"):
    start_part()
    peaks = np.round(np.arange(0.40, 0.80 + 1e-9, PEAK_STEP), 4); nP = len(peaks)
    rows = []
    for eps, inst, host, ncyc in [(0.01, False, "eps=0.01", 67 if not FAST else 40), (0.1, True, "instant", 20)]:
        m = ext(REF, eps, inst, rho=np.repeat(RHO, nP))
        conv = ratchet_ext(m, np.tile(peaks, nR), ncyc).reshape(nR, nP)
        for i, rho in enumerate(RHO):
            rows.append(dict(host=host, rho=rho, threshold=threshold_from(peaks, conv[i]), cycles=ncyc))
    A4 = pd.DataFrame(rows)
    piv = A4.pivot_table(index="rho", columns="host", values="threshold")
    piv["margin"] = piv["eps=0.01"] / piv["instant"]
    save(A4, P, "A4_ratchet_threshold.csv"); save(piv.reset_index(), P, "A4_ratchet_margin.csv")
    print(piv.round(4).to_string())
    ref_i = pm.ratchet_threshold(REF, 0.1, True, 60.0, peaks, 20)
    got = float(A4[(A4.host == "instant") & (A4.rho == 1.0)].threshold.iloc[0]) if 1.0 in RHOS else np.nan
    check(P, "rho=1 instant threshold equals pm.ratchet_threshold", (got, float(ref_i)), got == float(ref_i))
    mark_done(P, "A4")

## 17B. Pro-resolving adjunct after therapy

The adjunct is a course of 28 days during which dH/dt gains −λH (time 0 = end of plaque control, about one week after debridement). ρ = 1.

**B1. Relapse.** Treated sites at baselines from 0.15 to the static fold; no course, or a course of λ ∈ LAMS starting STARTS days after therapy. Recorded per site: relapse after SETTLE units and the time of the first H > 0.5; per arm: the relapse ceiling. Hosts ε = 0.03 and 0.01, r = 1 and 0.03, so the adjunct (host-directed) and slow recolonization (microbial) form a 2 × 2 at every λ.

**B2. Challenge in the window.** A 21-day challenge at the host's U\* (naive peak 0.30) at DAYS after therapy, for treated and naive sites; arms: no course; a course concurrent with the challenge (starting at its onset); an early course (days 0–28) followed by the challenge. λ = 0.5 and 1. The benefit of an arm is the reduction in peak H and the conversions it prevents.

In [ ]:
P = "17B_adjunct"
L_U = pm.days_to_units(COURSE_DAYS)
if not done(P, "B1"):
    start_part()
    fold = pm.folds(REF)[1]
    ubs = np.round(np.arange(0.15, fold + 1e-9, UB_STEP), 4); nU = len(ubs)
    arms = [("none", 0.0, 0.0)] + [(f"lam={l}, start={int(s)}d", l, s) for l in LAMS for s in STARTS]
    nA = len(arms)
    rows, crows = [], []
    for r in R_LIST:
        p = dict(REF, r=r)
        his = [eq_state_B(p, u, "high", 0.0) for u in ubs]
        hist = tuple(np.array([h[k] for h in his]) for k in range(5))
        for eps in EPS_LIST:
            m0 = ext(p, eps)
            st, _, _, _ = therapy5(m0, hist)
            sts = tuple(np.tile(x, nA) for x in st)
            lam0 = np.repeat([a[1] for a in arms], nU)
            s0 = np.repeat([pm.days_to_units(a[2]) for a in arms], nU)
            m = ext(p, eps, lam_fun=course(s0, lam0, L_U))
            first = np.full(nA * nU, np.nan)
            def mon(t, H, M, A, D, B):
                new = np.isnan(first) & (H > 0.5)
                first[new] = t
            fin, _ = m.run(sts, pm.const(np.tile(ubs, nA)), 0.0, SETTLE, monitor=mon)
            rel = (fin[0] > 0.5).reshape(nA, nU); first_d = (first * pm.UNIT_DAYS).reshape(nA, nU)
            for a, (name, l, s) in enumerate(arms):
                crows.append(dict(r=r, eps=eps, arm=name, lam=l, start_days=s,
                                  relapse_ceiling=relapse_ceiling_from(ubs, rel[a])))
                for j, u in enumerate(ubs):
                    rows.append(dict(r=r, eps=eps, arm=name, lam=l, start_days=s, baseline=u,
                                     relapsed=bool(rel[a, j]), first_H_above_0p5_days=first_d[a, j]))
    B1 = save(pd.DataFrame(rows), P, "B1_relapse_by_site.csv")
    B1c = save(pd.DataFrame(crows), P, "B1_relapse_ceiling.csv")
    print(B1c.round(4).to_string())
    for (r, eps), g in B1c.groupby(["r", "eps"]):
        base = float(g[g.arm == "none"].relapse_ceiling.iloc[0])
        check(P, f"no course reproduces relapse ceiling without adjunct (r={r}, eps={eps})", base)
    mark_done(P, "B1")

In [ ]:
if not done(P, "B2"):
    start_part()
    rows = []
    lowst = eq_state_B(REF, UB_REF, "low", 0.0); hist = eq_state_B(REF, UB_REF, "high", 0.0)
    lams_b2 = [0.5, 1.0]
    for eps in EPS_LIST:
        Ustar = float(calibrate_vec(ext(REF, eps), tile5(lowst, 1), UB_REF)[0])
        tr, _, _, _ = therapy5(ext(REF, eps), tile5(hist, 1))
        tdays = pm.days_to_units(DAYS); nD = len(DAYS)
        arms = [("none", 0.0, None)] + [(f"concurrent lam={l}", l, "conc") for l in lams_b2] + \
               [(f"early lam={l}", l, "early") for l in lams_b2]
        sts, lam0, s0, tons, meta = [], [], [], [], []
        for hname, s_ in (("naive", tile5(lowst, 1)), ("treated", tr)):
            for name, l, kind in arms:
                for j, d in enumerate(DAYS):
                    sts.append(s_); lam0.append(l); tons.append(tdays[j])
                    s0.append(tdays[j] if kind == "conc" else 0.0)
                    meta.append((hname, name, l, kind, d))
        sts = cat5(sts)
        m = ext(REF, eps, lam_fun=course(np.array(s0), np.array(lam0), L_U))
        res = eg5(m, sts, UB_REF, np.array(tons), Ustar, settle=SETTLE)
        for k, (hname, name, l, kind, d) in enumerate(meta):
            rows.append(dict(eps=eps, U_star=Ustar, history=hname, arm=name, lam=l, days=d,
                             peak_H=res["peak"][k], persistent=bool(res["persistent"][k]),
                             A_at_end=res["state"][2][k]))
    B2 = pd.DataFrame(rows)
    base = B2[B2.arm == "none"][["eps", "history", "days", "peak_H", "persistent"]].rename(
        columns=dict(peak_H="peak_none", persistent="persistent_none"))
    B2 = B2.merge(base, on=["eps", "history", "days"])
    B2["peak_reduction"] = B2.peak_none - B2.peak_H
    B2["conversion_prevented"] = B2.persistent_none & ~B2.persistent
    save(B2, P, "B2_challenge_with_adjunct.csv")
    print(B2[B2.history == "treated"].pivot_table(index=["eps", "arm"], columns="days", values="peak_H").round(2).to_string())
    mark_done(P, "B2")

## 17C. Community memory

Combinations: host ∈ {ε = 0.03, ε = 0.01, instant} × community ∈ {instant, μ ∈ MUS}; all share the equilibria of a given β (checked). β ∈ BETAS, r ∈ R_LIST, baseline 0.20.

**C1.** Folds and the second-root gap of M = G(H; K e^{−βM}); therapy (H at the end of plaque control, A and B retained, restored after SETTLE); relapse ceiling.

**C2.** Treated sites are run at baseline to each day in DAYS (snapshot). Each snapshot is compared with (i) a naive site and (ii) a *composition-matched* naive site: the naive equilibrium with B set to the treated site's B at that moment. All three receive the 21-day challenge at the combination's U\* (naive peak 0.30), and the critical step challenge, both starting at the snapshot. The step challenge lasts 2/ε of the host-memory group (2/0.03 or 2/0.01); the instant host is evaluated in both groups, so within a group every combination meets the same challenge. Outputs: peak H and persistence, critical challenge, treated/naive and treated/matched ratios. The decomposition table takes, at each day, the deficit d = 1 − (critical treated / critical naive) of host memory alone (memory host, instant community), community memory alone (instant host, μ) and both, and reports interaction = d_both − d_host − d_community. A treated site that has relapsed before the snapshot counts as d = 1.

In [ ]:
P = "17C_community_memory"
def combos_for():
    out = []
    for eps, inst, host in HOSTS:
        out.append(dict(host=host, eps=eps, inst=inst, community="instant", mu=0.03, icom=True))
        for mu in MUS:
            out.append(dict(host=host, eps=eps, inst=inst, community=f"mu={mu}", mu=mu, icom=False))
    return out
COMBOS = combos_for(); nC = len(COMBOS)

def combo_model(p, beta, rep=1, tile=1, combos=None, **kw):
    combos = COMBOS if combos is None else combos
    arr = lambda key: np.tile(np.repeat([c[key] for c in combos], rep), tile)
    return ExtModel(**p, eps=arr("eps"), instant=arr("inst").astype(bool), beta=beta, mu=arr("mu"),
                    instant_community=arr("icom").astype(bool), **kw)

if not done(P, "C1"):
    start_part()
    rows, frows, crows = [], [], []
    fold0 = pm.folds(REF)[1]
    for beta in BETAS:
        f, gap = folds_B(REF, beta)
        frows.append(dict(beta=beta, u_up=None if f is None else f[1], u_down=None if f is None else f[3],
                          second_root_gap=gap))
        check(P, f"beta={beta}: folds (u_up, u_down), gap", (None if f is None else (round(f[1], 4), round(f[3], 4)), gap),
              f is not None and gap < 1e-6)
        for r in R_LIST:
            p = dict(REF, r=r)
            for br in ("low", "high"):
                e = eq_state_B(p, UB_REF, br, beta)
                m = combo_model(p, beta)
                out, _ = m.run(tile5(e, nC), pm.const(UB_REF), 0.0, 400.0)
                dev = float(np.max(np.abs(out[0] - e[0])))
                check(P, f"beta={beta}, r={r}: every combination holds the {br} equilibrium", dev, dev < 1e-6)
            hist = eq_state_B(p, UB_REF, "high", beta)
            m = combo_model(p, beta)
            st, Hend, retA, retB = therapy5(m, tile5(hist, nC))
            fin, _ = m.run(st, pm.const(UB_REF), 0.0, SETTLE)
            for i, c in enumerate(COMBOS):
                rows.append(dict(beta=beta, r=r, host=c["host"], community=c["community"],
                                 H_after_plaque_control=Hend[i], A_retained=retA[i], B_retained=retB[i],
                                 B_disease=hist[4], restored=bool(fin[0][i] < 0.5)))
            ubs = np.round(np.arange(0.15, (f[1] if f else fold0) + 1e-9, max(UB_STEP, 0.01)), 4); nU = len(ubs)
            his = [eq_state_B(p, u, "high", beta) for u in ubs]
            hs = tuple(np.tile(np.array([h[k] for h in his]), nC) for k in range(5))
            m = combo_model(p, beta, rep=nU)
            st, _, _, _ = therapy5(m, hs)
            fin, _ = m.run(st, pm.const(np.tile(ubs, nC)), 0.0, SETTLE)
            rel = (fin[0] > 0.5).reshape(nC, nU)
            for i, c in enumerate(COMBOS):
                crows.append(dict(beta=beta, r=r, host=c["host"], community=c["community"],
                                  relapse_ceiling=relapse_ceiling_from(ubs, rel[i])))
    save(pd.DataFrame(frows), P, "C1_folds.csv")
    C1 = save(pd.DataFrame(rows), P, "C1_therapy.csv")
    C1c = save(pd.DataFrame(crows), P, "C1_relapse_ceiling.csv")
    print(C1.pivot_table(index=["beta", "r", "host"], columns="community", values="restored").to_string())
    print(C1c.pivot_table(index=["beta", "r", "host"], columns="community", values="relapse_ceiling").round(3).to_string())
    mark_done(P, "C1")

In [ ]:
if not done(P, "C2"):
    start_part()
    rows = []
    nD = len(DAYS); tdays = pm.days_to_units(DAYS)
    for beta in BETAS:
        for r in R_LIST:
            p = dict(REF, r=r)
            lowst = eq_state_B(p, UB_REF, "low", beta); hist = eq_state_B(p, UB_REF, "high", beta)
            m1 = combo_model(p, beta)
            Ustar = calibrate_vec(m1, tile5(lowst, nC), UB_REF)
            tr, _, _, _ = therapy5(m1, tile5(hist, nC))
            # snapshots of treated sites at each day
            snaps, t, st = [], 0.0, tr
            for d in tdays:
                if d > t:
                    st, _ = m1.run(st, pm.const(UB_REF), t, pm.align(d)); t = pm.align(d)
                snaps.append(tuple(np.array(x, float).copy() for x in st))
            T_ = tuple(np.concatenate([[s[k][i] for s in snaps] for i in range(nC)]) for k in range(5))  # combo-major
            N_ = tile5(lowst, nC * nD)
            Mt = (N_[0], N_[1], N_[2], N_[3], T_[4].copy())                                             # B matched
            sts = cat5([T_, N_, Mt])
            m3 = combo_model(p, beta, rep=nD, tile=3)
            U3 = np.tile(np.repeat(Ustar, nD), 3)
            pk = eg5(m3, sts, UB_REF, 0.0, U3, settle=SETTLE)
            K3 = nC * nD
            # critical step challenge, one duration (2/eps) per host-memory group; the instant host
            # belongs to both groups, so host-alone, community-alone and both share the same challenge
            for eps_g in EPS_LIST:
                idx = [i for i, c in enumerate(COMBOS) if c["host"] in (f"eps={eps_g}", "instant")]
                sel = np.concatenate([np.arange(i * nD, (i + 1) * nD) for i in idx]); nS = sel.size
                sel3 = np.concatenate([sel + b * K3 for b in range(3)])
                msub = combo_model(p, beta, rep=nD, tile=3, combos=[COMBOS[i] for i in idx])
                c_, clo = pm.critical_challenge(msub, tuple(x[sel3] for x in sts), UB_REF, np.zeros(3 * nS),
                                               np.full(3 * nS, 2 / eps_g), SETTLE, n_iter=NBIS)
                for a, i in enumerate(idx):
                    c = COMBOS[i]
                    for j, d in enumerate(DAYS):
                        q, k = a * nD + j, i * nD + j
                        cT = UB_REF if clo[q] else c_[q]
                        cN, cM = c_[nS + q], c_[2 * nS + q]
                        rows.append(dict(beta=beta, r=r, challenge=f"2/{eps_g}", host=c["host"], community=c["community"],
                                         days=d, U_star=Ustar[i], B_treated=T_[4][k], A_treated=T_[2][k],
                                         relapsed_before_challenge=bool(T_[0][k] > 0.5),
                                         peak_treated=pk["peak"][k], peak_naive=pk["peak"][K3 + k],
                                         peak_matched=pk["peak"][2 * K3 + k],
                                         persistent_treated=bool(pk["persistent"][k]),
                                         persistent_naive=bool(pk["persistent"][K3 + k]),
                                         persistent_matched=bool(pk["persistent"][2 * K3 + k]),
                                         crit_treated=cT, crit_naive=cN, crit_matched=cM,
                                         ratio_naive=cT / cN, ratio_matched=cT / cM))
    C2 = save(pd.DataFrame(rows), P, "C2_window_and_matching.csv")
    print(C2[C2.days.isin([0, 90])].pivot_table(index=["challenge", "beta", "r", "host", "days"], columns="community",
                                                 values="ratio_naive").round(3).to_string())
    ctl = C2[(C2.host == "instant") & (C2.community == "instant")]
    check(P, "control (instant host, instant community): treated/naive = 1", ctl.ratio_naive.round(3).unique().tolist(),
          bool(np.allclose(ctl.ratio_naive, 1.0, atol=0.01)))
    # decomposition (within each challenge group)
    drows = []
    for (ch, beta, r, d), g in C2.groupby(["challenge", "beta", "r", "days"]):
        G = {(h, c_): 1.0 - v for h, c_, v in zip(g.host, g.community, g.ratio_naive)}
        d0 = G[("instant", "instant")]
        for hm in [h for h in ("eps=0.03", "eps=0.01") if ch == "2/" + h[4:]]:
            for mu in MUS:
                cm = f"mu={mu}"
                dh, dc, db = G[(hm, "instant")] - d0, G[("instant", cm)] - d0, G[(hm, cm)] - d0
                drows.append(dict(challenge=ch, beta=beta, r=r, days=d, host=hm, community=cm, deficit_host=dh,
                                  deficit_community=dc, deficit_both=db, interaction=db - dh - dc))
    C2d = save(pd.DataFrame(drows), P, "C2_decomposition.csv")
    print(C2d[C2d.days.isin([0, 90])].round(3).to_string())
    mark_done(P, "C2")

## Figures (quick diagnostics) and export

In [ ]:
figd = part_dir("figures")
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
try:
    h = load("17A_resolution_rate", "A2_half_life.csv")
    for (r, host), g in h.groupby(["r", "host"]):
        ax[0].plot(g.rho, g.ratio_immediately, "o-", label=f"{host}, r={r}")
    ax[0].set_xscale("log"); ax[0].set_xlabel("resolution rate ρ"); ax[0].set_ylabel("treated / naive (2/ε)")
    ax[0].set_title("17A window after therapy"); ax[0].legend(fontsize=6)
    mg = load("17A_resolution_rate", "A4_ratchet_margin.csv")
    ax[1].plot(mg.rho, mg.margin, "o-"); ax[1].set_xscale("log"); ax[1].axhline(1, color="k", lw=0.5)
    ax[1].set_xlabel("ρ"); ax[1].set_ylabel("ratchet margin (ε=0.01 / instant)"); ax[1].set_title("17A ratchet, T = 60")
except FileNotFoundError:
    pass
try:
    b2 = load("17B_adjunct", "B2_challenge_with_adjunct.csv")
    g = b2[(b2.history == "treated") & (b2.arm != "none")]
    for (eps, arm), gg in g.groupby(["eps", "arm"]):
        ax[2].plot(gg.days, gg.peak_reduction, "o-", label=f"ε={eps} {arm}")
    ax[2].set_xlabel("days since therapy"); ax[2].set_ylabel("peak H reduction"); ax[2].set_title("17B adjunct benefit")
    ax[2].legend(fontsize=5)
except FileNotFoundError:
    pass
try:
    cd = load("17C_community_memory", "C2_decomposition.csv")
    g = cd[(cd.days == 0) & (cd.host == "eps=0.01")]
    lab = [f"β={b}, r={r}, {c}" for b, r, c in zip(g.beta, g.r, g.community)]
    x = np.arange(len(g))
    ax[3].bar(x - 0.27, g.deficit_host, 0.27, label="host"); ax[3].bar(x, g.deficit_community, 0.27, label="community")
    ax[3].bar(x + 0.27, g.deficit_both, 0.27, label="both")
    ax[3].set_xticks(x); ax[3].set_xticklabels(lab, rotation=75, fontsize=5); ax[3].set_ylabel("deficit 1 − treated/naive")
    ax[3].set_title("17C day 0, ε = 0.01"); ax[3].legend(fontsize=6)
except FileNotFoundError:
    pass
fig.tight_layout(); fig.savefig(os.path.join(figd, "fig17_overview.png"), dpi=150)

saved = [pd.read_csv(f_) for f_ in sorted(glob.glob(os.path.join(OUT, "*", "checks_*.csv")))]
unsaved = pd.DataFrame([c_ for c_ in CHECKS if not c_.get("saved")], columns=["module", "check", "value", "passed"])
pd.concat(saved + [unsaved], ignore_index=True).to_csv(os.path.join(OUT, "all_checks.csv"), index=False)
with open(os.path.join(OUT, "README.txt"), "w") as f:
    f.write('''Experiment 17 - resolution kinetics, pro-resolving adjunct, community memory (see notebook markdown)
0_reduction: ExtModel reduces to periomemory 1.0.0 (trajectories, equilibria, Experiment_14 anchors).
17A_resolution_rate: A1 one gingivitis episode by rho; A2 therapy, window vs recovery, half-life, instant host,
  relapse ceiling (r = 1, 0.03); A3 calibrated rechallenge; A4 ratchet threshold and margin at T = 60.
17B_adjunct: B1 relapse by baseline and relapse ceiling with a 28-day clearance course (lambda, start), r = 1, 0.03;
  B2 21-day challenge with no course, concurrent course or early course.
17C_community_memory: C1 folds, therapy, retention, relapse ceiling for host x community memory; C2 critical
  plateau and rechallenge against naive and composition-matched naive sites; C2_decomposition host/community/interaction.
all_checks.csv: reduction checks, anchors and internal checks.
''')
print(f"elapsed {time.time() - T0:.0f} s")
zip_path = shutil.make_archive(OUT, "zip", os.path.dirname(os.path.abspath(OUT)), os.path.basename(OUT))
print("wrote", zip_path)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("download did not start; the zip is in Drive:", e)